# Formeln prüfen (ausserhalb der Notebook-Reihe)

**Ziel.** Prüfen, ob die Formeln des Projekts **richtig umgesetzt** sind. Die Formeln stehen in `01b_methodik_leitfaden`, Abschnitt 12 (gültiger Stand) und Abschnitt 7 (früherer Plan).

**Warum.** Ein Rechenfehler in einer Formel verfälscht jedes Ergebnis, ohne dass man es ihm ansieht. Darum wird jede selbst geschriebene Formel ein zweites Mal gerechnet. Das Prinzip ist immer gleich:

1. ein kleines Beispiel mit erfundenen Zahlen;
2. rechnen mit **unserem Code** (derselbe Wortlaut wie im genannten Notebook);
3. rechnen mit einer **anerkannten Bibliothek** (scikit-learn, SciPy, PyTorch, Shapely, rasterio) oder von Hand;
4. vergleichen: Stimmen beide Zahlen überein, ist unsere Umsetzung korrekt.

In **Prüfung 1** sind diese vier Schritte einzeln ausgeführt, je in einer Zelle. Die Prüfungen 2 bis 18 gehen genau gleich vor, stehen aber je in einer einzigen Zelle: zuerst das Beispiel, dann unser Code, dann die Bibliothek, am Schluss der Vergleich.

**Was das Notebook nicht prüft.** Ob die *Wahl* einer Methode sinnvoll ist, kann kein Programm bestätigen; das beurteilen Menschen (Betreuung, Begutachtung). Hier geht es nur um Rechenfehler.

**Eingaben:** keine. Das Notebook braucht keine Projektdaten und läuft in wenigen Sekunden.
**Ausgaben:** nur die Tabelle am Schluss (eine Zeile je Prüfung).

| Nr. | Formel | unser Code steht in | Vergleich mit | in diesem Durchgang verwendet? |
| --- | --- | --- | --- | --- |
| 1 | IoU, Genauigkeit, Trefferquote | Notebooks 07, 09, 10 | scikit-learn | ja |
| 2 | Mittlerer Fehler (MAE) und Bias | Notebook 10 | scikit-learn | ja |
| 3 | Bootstrap-Intervall, gepaart, über einzelne Einheiten | früherer Plan (`01b`, 7.9) | SciPy | Vorstufe; heute gilt Prüfung 17 |
| 4 | Kreuzentropie mit "nicht bewertet" (255) | Notebook 07 | Formel von Hand gegen PyTorch | ja |
| 5 | Vertrauensgrenze: wie viele Lücken braucht es? | `01b`, 7.8 | SciPy | nein: Konzept Lückenschluss, nicht angewendet |
| 6 | Entscheidungsregel für die Sicherheit 95 von 100 | `01b`, 7.8 | Durchrechnen aller Fälle | nein: Konzept Lückenschluss |
| 7 | Rundung Fläche → Anzahl Plätze | Notebooks 05, 09 | Definition "kaufmännisch runden" | ja |
| 8 | Vereinfachung nach Douglas–Peucker | früherer Plan (`01b`, 7.6) | gemessene Abweichung | nein: entfällt (`01b`, Abschnitt 11, E7) |
| 9 | Versetztes Rechteck: falsch beschrifteter Anteil | `01b`, 7.2 | Shapely | Begründung für die eigene Ground Truth |
| 10 | 10-cm-Bild als Mittel von 2 × 2 Pixeln | Notebooks 05, 08 | NumPy gegen rasterio | ja |
| 11 | Pflanzenindex NDVI | Notebook 01, `ground_truth_erfassen` | Rechnung von Hand | nur für die Vorschläge beim Erfassen |
| 12 | Platzfläche passt zum Lösungsblatt | `01b`, Abschnitte 1 und 12.2 | Shapely | ja |
| 13 | Runden oder abrunden? | `01b`, 8.1 K3 | Simulation | ja |
| 14 | Schwelle der Sicherheit und Unterschied der Rohwerte | Notebook 08 | PyTorch | ja |
| 15 | Mittlere Tiefe einer Fläche und ihre Klasse | Notebooks 05, 09 | Rechtecke mit bekannter Tiefe (Shapely) | ja |
| 16 | Platz-Äquivalente, einmal je Ausschnitt gerundet | Notebooks 05, 09 | Rechnung von Hand | ja |
| 17 | Bootstrap über ganze Blöcke | Notebook 10 | dieselben Ziehungen Schritt für Schritt | ja |
| 18 | Zerlegung des Deltas in drei Teile | Notebook 10 | Summe der Teile | ja |

In [1]:
import numpy as np                 # Rechnen mit Zahlenfeldern
import pandas as pd                # Tabellen

rng = np.random.default_rng(42)    # fester Zufall: bei jedem Lauf dieselben Beispielzahlen
protokoll = []                     # Sammelliste: eine Zeile je Prüfung

def vergleiche(name, unser, bibliothek, toleranz=1e-9):
    # Zwei Zahlen vergleichen und das Ergebnis ins Protokoll schreiben.
    # toleranz = wie weit die Zahlen auseinanderliegen dürfen (Rundung im Rechner)
    gleich = bool(abs(unser - bibliothek) <= toleranz)
    protokoll.append(dict(pruefung=name, unser_code=unser, bibliothek=bibliothek, stimmt="✅" if gleich else "❌"))
    print(f"{name}: unser Code = {unser:.6f} | Bibliothek = {bibliothek:.6f} -> {'✅ gleich' if gleich else '❌ VERSCHIEDEN'}")

## 1. IoU, Genauigkeit, Trefferquote

**In Worten.** Wir vergleichen die Antwort des Netzes mit dem Lösungsblatt, Pixel für Pixel. Pixel mit 255 werden nicht bewertet.
- **TP** = Netz sagt Parkfläche, stimmt. **FP** = Netz sagt Parkfläche, stimmt nicht. **FN** = Netz übersieht Parkfläche.
- IoU = TP ÷ (TP + FP + FN); Genauigkeit = TP ÷ (TP + FP); Trefferquote = TP ÷ (TP + FN).

**Schritt 1: das Beispiel.** 10'000 erfundene Pixel: ein Lösungsblatt mit etwa 10 % Parkfläche und eine Antwort des Netzes, die in 85 % der Pixel stimmt. 5 % der Pixel sind nicht bewertet (255).

Vorher: nichts → nachher: zwei Reihen von 10'000 Zahlen (`loesung`, `antwort`).

In [2]:
loesung = (rng.random(10_000) < 0.10).astype(int)          # Lösungsblatt: 1 = Parkfläche (etwa 10 %)
antwort = np.where(rng.random(10_000) < 0.85, loesung, 1 - loesung)   # Netz: in 85 % der Pixel richtig, sonst umgekehrt
loesung[rng.random(10_000) < 0.05] = 255                    # 5 % der Pixel: nicht bewertet
print(f"{len(loesung)} Pixel | Parkfläche laut Lösung: {(loesung == 1).sum()} | nicht bewertet: {(loesung == 255).sum()} | Netz meldet Parkfläche: {(antwort == 1).sum()}")

10000 Pixel | Parkfläche laut Lösung: 918 | nicht bewertet: 518 | Netz meldet Parkfläche: 2195


**Schritt 2: rechnen mit unserem Code.** Derselbe Wortlaut wie in Notebook 07 (Funktionen `konfusion` und `kennzahlen`): zuerst TP, FP und FN zählen, dann die drei Kennzahlen.

Vorher: Lösung und Antwort je Pixel → nachher: drei Zahlen (IoU, Genauigkeit, Trefferquote).

In [3]:
bewertbar = loesung != 255
tp = ((antwort == 1) & (loesung == 1) & bewertbar).sum()
fp = ((antwort == 1) & (loesung == 0) & bewertbar).sum()
fn = ((antwort == 0) & (loesung == 1) & bewertbar).sum()
unser = dict(iou=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))
print(f"TP = {tp}, FP = {fp}, FN = {fn} → IoU {unser['iou']:.6f}, Genauigkeit {unser['precision']:.6f}, Trefferquote {unser['recall']:.6f}")

TP = 786, FP = 1287, FN = 132 → IoU 0.356463, Genauigkeit 0.379161, Trefferquote 0.856209


**Schritt 3: rechnen mit der Bibliothek.** scikit-learn kennt dieselben drei Kennzahlen (die IoU heisst dort `jaccard_score`). Die Bibliothek kennt "nicht bewertet" nicht; sie bekommt darum nur die bewertbaren Pixel.

Vorher: dieselben Pixel → nachher: dieselben drei Kennzahlen, von fremdem Code gerechnet.

In [4]:
from sklearn.metrics import jaccard_score, precision_score, recall_score   # jaccard = anderer Name für IoU

l, a = loesung[bewertbar], antwort[bewertbar]              # nur die bewertbaren Pixel
bibliothek = dict(iou=jaccard_score(l, a), precision=precision_score(l, a), recall=recall_score(l, a))
print(f"IoU {bibliothek['iou']:.6f}, Genauigkeit {bibliothek['precision']:.6f}, Trefferquote {bibliothek['recall']:.6f}")

IoU 0.356463, Genauigkeit 0.379161, Trefferquote 0.856209


**Schritt 4: vergleichen.** Die Funktion `vergleiche` stellt je zwei Zahlen nebeneinander und schreibt eine Zeile ins Protokoll. ✅ heisst: Sie stimmen bis auf die Rundung im Rechner überein.

In [5]:
vergleiche("1 IoU", unser["iou"], bibliothek["iou"])
vergleiche("1 Genauigkeit", unser["precision"], bibliothek["precision"])
vergleiche("1 Trefferquote", unser["recall"], bibliothek["recall"])

1 IoU: unser Code = 0.356463 | Bibliothek = 0.356463 -> ✅ gleich
1 Genauigkeit: unser Code = 0.379161 | Bibliothek = 0.379161 -> ✅ gleich
1 Trefferquote: unser Code = 0.856209 | Bibliothek = 0.856209 -> ✅ gleich


## 2. Mittlerer Fehler (MAE) und Bias

**In Worten.** Je Ausschnitt: gezählt minus wahr. MAE = Mittel der Beträge dieser Fehler. Bias = (Summe gezählt − Summe wahr) ÷ Summe wahr, in Prozent: negativ heisst zu wenig gezählt.

**Beispiel:** 96 erfundene Ausschnitte. Notebook 10 gibt den Bias als Anteil aus (ohne × 100); die Formel ist dieselbe.

In [6]:
from sklearn.metrics import mean_absolute_error

t = pd.DataFrame({"wahr": rng.poisson(4, 96)})                          # wahre Anzahl je Ausschnitt
t["modell"] = np.maximum(t.wahr + rng.integers(-3, 3, 96), 0).astype(float)   # gezählt: wahr mit Fehler, nie negativ
t.loc[::10, "modell"] = np.nan                                          # jeder 10. Ausschnitt: nichts gemeldet (leer)

# --- unser Code (wie in Notebook 10, Abschnitt 3)
gezaehlt = t["modell"].fillna(0)                                        # leer = 0 Plätze gezählt
unser_mae = (gezaehlt - t.wahr).abs().mean()
unser_bias = 100 * (gezaehlt.sum() - t.wahr.sum()) / t.wahr.sum()

# --- Vergleich
vergleiche("2 MAE", unser_mae, mean_absolute_error(t.wahr, gezaehlt))
von_hand = 100 * (sum(gezaehlt) / sum(t.wahr) - 1)                      # dieselbe Formel, anders hingeschrieben
vergleiche("2 Bias in %", unser_bias, von_hand)

2 MAE: unser Code = 1.593750 | Bibliothek = 1.593750 -> ✅ gleich
2 Bias in %: unser Code = -20.053476 | Bibliothek = -20.053476 -> ✅ gleich


## 3. Bootstrap-Intervall (gepaart)

**In Worten.** Frage: Ist Verfahren A (z. B. 10 cm) schlechter als Verfahren B (5 cm), oder ist der Unterschied Zufall? Wir ziehen 10'000-mal eine neue Stichprobe von Einheiten (mit Zurücklegen) und rechnen jedes Mal den Unterschied der mittleren Fehler. Die mittleren 95 % dieser 10'000 Werte sind das Intervall. *Gepaart* heisst: Für A und B werden dieselben Abschnitte gezogen.

**Vergleich.** `scipy.stats.bootstrap` macht dasselbe, zieht aber andere Zufallszahlen. Die Grenzen können darum nicht auf alle Stellen gleich sein; sie müssen aber **nahe beieinander** liegen. Als Toleranz nehmen wir 0,05 Plätze.

In [7]:
from scipy.stats import bootstrap as scipy_bootstrap

fehler_a = np.abs(rng.normal(3.0, 2.0, 96))          # Betrag des Fehlers je Einheit, Verfahren A
fehler_b = np.abs(fehler_a - rng.normal(1.0, 1.5, 96))   # Verfahren B: an derselben Einheit meist etwas besser

# --- unser Code (Ziehung einzelner Einheiten, wie im früheren Plan; Notebook 10 zieht ganze Blöcke: Prüfung 17)
B = 10_000
zufall = np.random.default_rng(42)
n = len(fehler_a)
ziehungen = zufall.integers(0, n, size=(B, n))                          # 10'000 Stichproben auf einmal
differenzen = fehler_a[ziehungen].mean(axis=1) - fehler_b[ziehungen].mean(axis=1)
unten, oben = np.quantile(differenzen, [0.025, 0.975])

# --- Bibliothek
def unterschied(a, b, axis=-1):
    return a.mean(axis=axis) - b.mean(axis=axis)                        # dieselbe Kennzahl wie oben

ergebnis = scipy_bootstrap((fehler_a, fehler_b), unterschied, paired=True, n_resamples=B,
                           confidence_level=0.95, method="percentile", random_state=1)
vergleiche("3 Bootstrap untere Grenze", unten, ergebnis.confidence_interval.low, toleranz=0.05)
vergleiche("3 Bootstrap obere Grenze", oben, ergebnis.confidence_interval.high, toleranz=0.05)

3 Bootstrap untere Grenze: unser Code = 0.444359 | Bibliothek = 0.447419 -> ✅ gleich
3 Bootstrap obere Grenze: unser Code = 1.061571 | Bibliothek = 1.071662 -> ✅ gleich


## 4. Kreuzentropie mit "nicht bewertet"

**In Worten.** Für jedes bewertete Pixel: Wie wahrscheinlich fand das Netz die **richtige** Antwort? Davon der Logarithmus, mit Minuszeichen; über alle bewerteten Pixel gemittelt. Sichere falsche Antworten kosten viel. Pixel mit 255 zählen nicht mit.

Im Projekt rechnet PyTorch diese Formel. Hier schreiben wir sie **von Hand** hin und prüfen, dass PyTorch dasselbe liefert – so wissen wir, dass die Formel in 01b (Abschnitt 7.4) wirklich das beschreibt, was gerechnet wird.

In [8]:
import torch
import torch.nn.functional as F

rohwerte = rng.normal(0, 2, size=(1, 2, 8, 8))              # Netz-Ausgabe: 1 Bild, 2 Klassen, 8 × 8 Pixel
loesung = rng.integers(0, 2, size=(1, 8, 8))                # Lösungsblatt: 0 oder 1
loesung[0, :2, :] = 255                                     # die obersten zwei Zeilen: nicht bewertet

# --- Formel von Hand
e = np.exp(rohwerte)
p_parkflaeche = e[:, 1] / (e[:, 0] + e[:, 1])               # Softmax: Wahrscheinlichkeit für "Parkfläche"
bewertbar = loesung != 255
y, p = loesung[bewertbar], p_parkflaeche[bewertbar]
von_hand = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# --- PyTorch (so wird es in Notebook 07 gerechnet)
pytorch = F.cross_entropy(torch.tensor(rohwerte), torch.tensor(loesung), ignore_index=255).item()
vergleiche("4 Kreuzentropie", von_hand, pytorch)

4 Kreuzentropie: unser Code = 1.176514 | Bibliothek = 1.176514 -> ✅ gleich


## 5. Wie viele geprüfte Lücken braucht es für "95 von 100"?

**In Worten.** Wir haben $N$ verdeckte Lücken geprüft, und in allen lagen wirklich Parkfelder. Wie sicher dürfen wir sein, dass das auch bei der nächsten Lücke stimmt? Die Statistik gibt eine **untere Grenze** (hier mit 95 % Vertrauen). Unsere Formel aus 01b: untere Grenze $= 0{,}05^{1/N}$, und daraus $N \ge \ln 0{,}05 \div \ln q_0$.

**Vergleich.** `scipy.stats.binomtest(...).proportion_ci(method="exact")` rechnet die Grenze nach Clopper–Pearson. Eine einseitige Grenze mit 95 % entspricht dort dem zweiseitigen Intervall mit 90 %.

In [9]:
import math
from scipy.stats import binomtest

for q0 in (0.90, 0.95, 0.99):
    n_min = math.ceil(math.log(0.05) / math.log(q0))                    # unsere Formel: kleinste nötige Anzahl
    unser = 0.05 ** (1 / n_min)                                         # unsere Formel: untere Grenze bei n_min Lücken
    scipy_grenze = binomtest(n_min, n_min).proportion_ci(confidence_level=0.90, method="exact").low
    vergleiche(f"5 untere Grenze, {n_min} Lücken (Ziel {q0})", unser, scipy_grenze)
    # Gegenprobe: mit einer Lücke weniger darf das Ziel NICHT erreicht sein
    zu_wenig = binomtest(n_min - 1, n_min - 1).proportion_ci(confidence_level=0.90, method="exact").low
    print(f"   Ziel {q0}: {n_min} Lücken reichen ({scipy_grenze:.5f} ≥ {q0}); {n_min - 1} reichen nicht ({zu_wenig:.5f} < {q0}): {zu_wenig < q0 <= scipy_grenze}")

5 untere Grenze, 29 Lücken (Ziel 0.9): unser Code = 0.901855 | Bibliothek = 0.901855 -> ✅ gleich
   Ziel 0.9: 29 Lücken reichen (0.90186 ≥ 0.9); 28 reichen nicht (0.89853 < 0.9): True
5 untere Grenze, 59 Lücken (Ziel 0.95): unser Code = 0.950492 | Bibliothek = 0.950492 -> ✅ gleich
   Ziel 0.95: 59 Lücken reichen (0.95049 ≥ 0.95); 58 reichen nicht (0.94966 < 0.95): True
5 untere Grenze, 299 Lücken (Ziel 0.99): unser Code = 0.990031 | Bibliothek = 0.990031 -> ✅ gleich
   Ziel 0.99: 299 Lücken reichen (0.99003 ≥ 0.99); 298 reichen nicht (0.99000 < 0.99): True


## 6. Entscheidungsregel: ab welcher Sicherheit lohnt sich Ergänzen?

**In Worten.** Ein erfundener Platz soll $c_{+}$-mal so schwer wiegen wie ein übersehener. Formel aus 01b: Ergänzen ist besser, sobald die Sicherheit $q$ über $q_0 = c_{+} \div (c_{+} + 1)$ liegt. Für "19-mal so schwer" ergibt das 0,95.

**Prüfung.** Wir rechnen für viele Werte von $q$ beide erwarteten Kosten aus und suchen die Stelle, an der Ergänzen erstmals billiger ist.

In [10]:
for c_plus in (9, 19, 99):
    q = np.linspace(0, 1, 100_001)                       # alle Sicherheiten von 0 bis 1 in feinen Schritten
    kosten_ergaenzen = (1 - q) * c_plus                  # mit Wahrscheinlichkeit 1-q erfunden, kostet c_plus
    kosten_weglassen = q * 1                             # mit Wahrscheinlichkeit q übersehen, kostet 1
    wechsel = q[np.argmax(kosten_ergaenzen < kosten_weglassen)]   # erste Sicherheit, bei der Ergänzen billiger ist
    vergleiche(f"6 Wechselpunkt bei c+ = {c_plus}", c_plus / (c_plus + 1), wechsel, toleranz=2e-5)

6 Wechselpunkt bei c+ = 9: unser Code = 0.900000 | Bibliothek = 0.900000 -> ✅ gleich
6 Wechselpunkt bei c+ = 19: unser Code = 0.950000 | Bibliothek = 0.950000 -> ✅ gleich
6 Wechselpunkt bei c+ = 99: unser Code = 0.990000 | Bibliothek = 0.990000 -> ✅ gleich


## 7. Rundung: von der Fläche zur Anzahl Plätze

**In Worten.** Anzahl = Fläche ÷ Platzfläche, gerundet. **Kaufmännisch** runden heisst: Bei genau ,5 wird aufgerundet (2,5 → 3). Die Notebooks 05 und 09 rechnen `np.floor(x + 0.5)`. Wir prüfen das gegen die Definition; Python kennt sie als `ROUND_HALF_UP`.

**Zum Vergleich** steht `np.rint` daneben. Es wurde im ersten Pilot verwendet und rundet bei genau ,5 zur nächsten **geraden** Zahl.

In [11]:
from decimal import Decimal, ROUND_HALF_UP                             # Rechnen mit Dezimalzahlen und festen Rundungsregeln

werte = np.array([0.4, 0.5, 0.6, 1.5, 2.5, 3.5, 4.5])
tabelle = pd.DataFrame({"Fläche ÷ Platzfläche": werte,
                        "np.floor(x + 0.5) (Notebooks 05, 09)": np.floor(werte + 0.5).astype(int),               # unser Code
                        "kaufmännisch (Definition)": [int(Decimal(str(w)).quantize(Decimal("1"), rounding=ROUND_HALF_UP)) for w in werte],
                        "np.rint (Pilot 1)": np.rint(werte).astype(int)})                                        # nur zum Vergleich
abweichungen = int((tabelle.iloc[:, 1] != tabelle.iloc[:, 2]).sum())
vergleiche("7 Werte, bei denen unser Runden von der Definition abweicht", float(abweichungen), 0.0)
tabelle

7 Werte, bei denen unser Runden von der Definition abweicht: unser Code = 0.000000 | Bibliothek = 0.000000 -> ✅ gleich


,Fläche ÷ Platzfläche,"np.floor(x + 0.5) (Notebooks 05, 09)",kaufmännisch (Definition),np.rint (Pilot 1)
0,0.4,0,0,0
1,0.5,1,1,0
2,0.6,1,1,1
3,1.5,2,2,2
4,2.5,3,3,2
5,3.5,4,4,4
6,4.5,5,5,4


**Befund.** `np.floor(x + 0.5)` rundet kaufmännisch. `np.rint` täte das nicht (0,5 → 0; 2,5 → 2; 4,5 → 4); darum wird es nicht mehr verwendet.

## 8. Vereinfachung nach Douglas–Peucker

**Stand.** In diesem Durchgang werden Umrisse nicht mehr vereinfacht (`01b`, Abschnitt 11, E7). Die Prüfung bleibt stehen, weil die Formel im früheren Plan (`01b`, 7.6) beschrieben ist.

**In Worten.** Die Ränder der erkannten Flächen sind treppenförmig (Pixel). `simplify(toleranz)` entfernt Zwischenpunkte, solange die neue Linie höchstens `toleranz` von der alten abweicht. Wir prüfen diese Zusage: Die grösste Abweichung (Hausdorff-Abstand) darf nicht über der Toleranz liegen.

In [12]:
from shapely.geometry import LineString

x = np.linspace(0, 50, 400)
linie = LineString(np.c_[x, np.sin(x) + rng.normal(0, 0.05, 400)])     # wellige, leicht verrauschte Linie, 50 m lang
for toleranz in (0.10, 0.25, 0.50):
    einfach = linie.simplify(toleranz)                                 # Douglas–Peucker (wie im früheren Plan, 01b 7.6)
    abweichung = linie.hausdorff_distance(einfach)                     # grösster Abstand zwischen alter und neuer Linie
    ok = abweichung <= toleranz + 1e-9
    protokoll.append(dict(pruefung=f"8 Douglas–Peucker, Toleranz {toleranz} m", unser_code=abweichung, bibliothek=toleranz,
                          stimmt="✅" if ok else "❌"))
    print(f"Toleranz {toleranz:.2f} m: {len(linie.coords)} -> {len(einfach.coords)} Punkte, grösste Abweichung {abweichung:.3f} m -> {'✅ eingehalten' if ok else '❌'}")

Toleranz 0.10 m: 400 -> 64 Punkte, grösste Abweichung 0.100 m -> ✅ eingehalten
Toleranz 0.25 m: 400 -> 29 Punkte, grösste Abweichung 0.246 m -> ✅ eingehalten
Toleranz 0.50 m: 400 -> 18 Punkte, grösste Abweichung 0.405 m -> ✅ eingehalten


## 9. Versetztes Rechteck: Wie viel ist falsch beschriftet?

**In Worten.** Ein Feld der Länge $L$ wird um $\delta$ verschoben gezeichnet. Dann ist auf der einen Seite ein Streifen fälschlich "Parkplatz", auf der anderen ein Streifen fälschlich "kein Parkplatz". Formel aus 01b: falscher Anteil $= 2\delta \div L$ (bezogen auf die Feldfläche).

In [13]:
from shapely.geometry import box

L, tiefe = 2.5, 5.0                                         # Beispiel: Feld 2,5 m breit (Verschiebungsrichtung), 5 m tief
for delta in (0.25, 0.5, 1.0):
    echt = box(0, 0, L, tiefe)                              # das Feld im Luftbild
    versetzt = box(delta, 0, L + delta, tiefe)              # das gezeichnete Rechteck
    gemessen = echt.symmetric_difference(versetzt).area / echt.area    # Fläche, die nur in einem der beiden liegt
    vergleiche(f"9 falscher Anteil bei {delta} m Versetzung", 2 * delta / L, gemessen)

9 falscher Anteil bei 0.25 m Versetzung: unser Code = 0.200000 | Bibliothek = 0.200000 -> ✅ gleich
9 falscher Anteil bei 0.5 m Versetzung: unser Code = 0.400000 | Bibliothek = 0.400000 -> ✅ gleich
9 falscher Anteil bei 1.0 m Versetzung: unser Code = 0.800000 | Bibliothek = 0.800000 -> ✅ gleich


## 10. Das 10-cm-Bild als Mittel von 2 × 2 Pixeln

**In Worten.** Aus je vier 5-cm-Pixeln (2 × 2) wird ein 10-cm-Pixel: der Mittelwert. Im Projekt macht das rasterio (`Resampling.average`). Wir bauen ein kleines Bild im Arbeitsspeicher, lassen es von rasterio halbieren und rechnen dasselbe mit NumPy von Hand.

In [14]:
import rasterio
from rasterio.io import MemoryFile
from rasterio.enums import Resampling
from rasterio.transform import from_origin

klein = rng.integers(0, 256, size=(8, 8)).astype("float32")           # Bild 8 × 8 Pixel, Werte 0–255

# --- von Hand: je 2 × 2 Pixel mitteln
von_hand = klein.reshape(4, 2, 4, 2).mean(axis=(1, 3))                # 8×8 -> 4 Blöcke × 2 Pixel in jeder Richtung

# --- rasterio: Bild als Datei im Arbeitsspeicher, dann in halber Grösse lesen
with MemoryFile() as speicher:
    with speicher.open(driver="GTiff", height=8, width=8, count=1, dtype="float32",
                       transform=from_origin(0, 0.4, 0.05, 0.05)) as datei:     # 5-cm-Pixel
        datei.write(klein, 1)
    with speicher.open() as datei:
        von_rasterio = datei.read(1, out_shape=(4, 4), resampling=Resampling.average)   # 4 × 4 = 10-cm-Pixel

vergleiche("10 grösster Unterschied der 16 Pixel", float(np.abs(von_hand - von_rasterio).max()), 0.0, toleranz=1e-4)

10 grösster Unterschied der 16 Pixel: unser Code = 0.000000 | Bibliothek = 0.000000 -> ✅ gleich


## 11. Pflanzenindex NDVI

**In Worten.** NDVI = (Infrarot − Rot) ÷ (Infrarot + Rot). Pflanzen strahlen viel Infrarot und wenig Rot zurück → Wert nahe +1. Asphalt strahlt beides ähnlich zurück → Wert nahe 0. Der Wert liegt immer zwischen −1 und +1.

**Prüfung.** Drei Pixel von Hand, dazu die Grenzen für viele zufällige Pixel.

In [15]:
def ndvi(infrarot, rot):
    return (infrarot - rot) / (infrarot + rot + 1e-6)                  # wie in Notebook 01 (+ 1e-6 gegen Division durch 0)

vergleiche("11 NDVI Baum (Infrarot 200, Rot 50)", ndvi(200.0, 50.0), 150 / 250, toleranz=1e-6)      # = 0,6
vergleiche("11 NDVI Asphalt (Infrarot 90, Rot 100)", ndvi(90.0, 100.0), -10 / 190, toleranz=1e-6)   # ≈ −0,05
viele = ndvi(rng.integers(0, 256, 100_000).astype(float), rng.integers(0, 256, 100_000).astype(float))
im_bereich = bool((viele >= -1).all() and (viele <= 1).all())
protokoll.append(dict(pruefung="11 NDVI immer zwischen −1 und +1", unser_code=float(viele.min()), bibliothek=float(viele.max()),
                      stimmt="✅" if im_bereich else "❌"))
print("kleinster und grösster Wert bei 100'000 Pixeln:", viele.min().round(3), viele.max().round(3))

11 NDVI Baum (Infrarot 200, Rot 50): unser Code = 0.600000 | Bibliothek = 0.600000 -> ✅ gleich
11 NDVI Asphalt (Infrarot 90, Rot 100): unser Code = -0.052632 | Bibliothek = -0.052632 -> ✅ gleich
kleinster und grösster Wert bei 100'000 Pixeln: -1.0 1.0


## 12. Passt die Platzfläche zum Lösungsblatt?

**In Worten.** Anzahl = Fläche ÷ Platzfläche. Das geht nur auf, wenn ein Feld im Lösungsblatt genau die Platzfläche einnimmt. Wir zeichnen eine Reihe von 10 Querfeldern, die **2,33 m** auseinanderliegen und 5 m tief sind, und zählen die Fläche auf zwei Arten:

- **Variante a (Norm):** Kästchen 2,50 m breit, Platzfläche 2,50 × 5,00 m. Die Kästchen überlappen sich.
- **Variante b (Entscheid):** Kästchen so breit wie der Abstand (2,33 m), Platzfläche 2,33 × 5,00 m.

Erwartung aus 01b: Variante a zählt um den Faktor 2,33 ÷ 2,50 zu wenig (bis auf den Rand der Reihe), Variante b zählt genau 10.

In [16]:
from shapely.geometry import box
from shapely.ops import unary_union

abstand, tiefe, anzahl = 2.33, 5.00, 10
mitten = [i * abstand for i in range(anzahl)]                          # Mittelpunkte der 10 Felder entlang der Strasse

def gezaehlt(breite):
    kaestchen = [box(m - breite / 2, 0, m + breite / 2, tiefe) for m in mitten]   # ein Kästchen je Feld
    flaeche = unary_union(kaestchen).area                              # Fläche im Lösungsblatt (Überlappung zählt nur einmal)
    return flaeche / (breite * tiefe)                                  # Fläche ÷ Platzfläche

# Variante a: erwartet (9 Abstände + 1 volle Normbreite) ÷ Normbreite
vergleiche("12 Variante a (Norm 2,50 m): gezählt", gezaehlt(2.50), (9 * abstand + 2.50) / 2.50)
vergleiche("12 Variante b (Abstand 2,33 m): gezählt", gezaehlt(abstand), 10.0)
print("Variante a zählt", round(gezaehlt(2.50), 2), "statt 10 Plätze; Variante b zählt", round(gezaehlt(abstand), 2))

12 Variante a (Norm 2,50 m): gezählt: unser Code = 9.388000 | Bibliothek = 9.388000 -> ✅ gleich
12 Variante b (Abstand 2,33 m): gezählt: unser Code = 10.000000 | Bibliothek = 10.000000 -> ✅ gleich
Variante a zählt 9.39 statt 10 Plätze; Variante b zählt 10.0


## 13. Runden oder abrunden?

**In Worten.** In 01b stehen zwei Rundungen: Bei **gesehener** Fläche wird kaufmännisch gerundet, bei **erschlossenen** Lücken abgerundet. Wir prüfen die Begründung: Kaufmännisches Runden ist im Mittel neutral (Fehler 0), Abrunden zählt im Mittel einen halben Platz zu wenig.

**Prüfung.** 200'000 zufällige Werte "Fläche ÷ Platzfläche", jeweils gerundet und abgerundet; verglichen wird der mittlere Fehler mit dem Sollwert.

In [17]:
x = rng.uniform(0, 10, 200_000)                                        # zufällige Verhältnisse Fläche ÷ Platzfläche
fehler_runden = (np.floor(x + 0.5) - x).mean()                         # kaufmännisch runden: mittlerer Fehler
fehler_abrunden = (np.floor(x) - x).mean()                             # abrunden: mittlerer Fehler
vergleiche("13 mittlerer Fehler beim Runden", fehler_runden, 0.0, toleranz=0.005)
vergleiche("13 mittlerer Fehler beim Abrunden", fehler_abrunden, -0.5, toleranz=0.005)

13 mittlerer Fehler beim Runden: unser Code = -0.000857 | Bibliothek = 0.000000 -> ✅ gleich
13 mittlerer Fehler beim Abrunden: unser Code = -0.499777 | Bibliothek = -0.500000 -> ✅ gleich


## 14. Schwelle der Sicherheit und Unterschied der Rohwerte

**In Worten.** Das Netz gibt je Pixel zwei Rohwerte. Notebook 07 wählt die Schwelle $\tau$ für die **Sicherheit** $p$ (softmax). Notebook 08 mittelt aber den **Unterschied** $d$ der Rohwerte (Parkfläche − keine) und meldet ein Pixel, wenn $d > \ln\big(\tau / (1 - \tau)\big)$. Beides muss dieselbe Meldung geben, weil $p = 1 / (1 + e^{-d})$.

**Prüfung.** 1'000 erfundene Pixel, sechs Schwellen: Wie viele Pixel werden nach der einen Regel gemeldet und nach der anderen nicht?

In [18]:
import torch

rohwerte = torch.tensor(rng.normal(0, 3, size=(1000, 2)))              # zwei Rohwerte je Pixel: keine Parkfläche, Parkfläche
sicherheit = rohwerte.softmax(1)[:, 1].numpy()                         # Sicherheit für Parkfläche (PyTorch, wie in Notebook 07)
unterschied = (rohwerte[:, 1] - rohwerte[:, 0]).numpy()                # Unterschied der Rohwerte (wie in Notebook 08)
abweichend = 0
for schwelle in [0.10, 0.35, 0.40, 0.45, 0.50, 0.90]:
    grenze = np.log(schwelle / (1 - schwelle))                         # unser Code (Notebook 08)
    abweichend += int(((sicherheit > schwelle) != (unterschied > grenze)).sum())
vergleiche("14 Pixel mit verschiedener Meldung (Sicherheit gegen Unterschied)", float(abweichend), 0.0)

14 Pixel mit verschiedener Meldung (Sicherheit gegen Unterschied): unser Code = 0.000000 | Bibliothek = 0.000000 -> ✅ gleich


## 15. Mittlere Tiefe einer Fläche und ihre Klasse

**In Worten.** Die Zählung braucht für jede gemeldete Fläche die Klasse: `eng` (quer oder schräg, 5 m tief) oder `weit` (längs, 1,90 m tief). Die Notebooks 05 und 09 lesen sie aus der **mittleren Tiefe**: Fläche ÷ lange Seite des kleinsten umschliessenden Rechtecks. Liegt sie über 3,45 m, ist die Fläche `eng`.

**Prüfung.** Zwei Reihen mit bekannter Tiefe, schräg ins Bild gedreht: vier Querfelder nebeneinander (5 m tief) und drei Längsfelder hintereinander (1,90 m tief).

In [19]:
from shapely import affinity                                           # Formen drehen
from shapely.geometry import box                                       # Rechteck aus vier Zahlen

def mittlere_tiefe(flaeche):
    # unser Code (wie in den Notebooks 05 und 09): Fläche ÷ lange Seite des kleinsten umschliessenden Rechtecks
    ecken = np.array(flaeche.minimum_rotated_rectangle.exterior.coords)
    kanten = ecken[1:3] - ecken[0:2]                                   # seine zwei Kanten als Vektoren
    return flaeche.area / np.hypot(kanten[:, 0], kanten[:, 1]).max()

querreihe = affinity.rotate(box(0, 0, 4 * 2.5, 5.0), 33)               # 10 m lang, 5 m tief, um 33° gedreht
laengsreihe = affinity.rotate(box(0, 0, 3 * 5.6, 1.9), -20)            # 16,8 m lang, 1,9 m tief, um 20° gedreht
vergleiche("15 mittlere Tiefe der Querreihe (Soll 5,00 m)", mittlere_tiefe(querreihe), 5.0)
vergleiche("15 mittlere Tiefe der Längsreihe (Soll 1,90 m)", mittlere_tiefe(laengsreihe), 1.9)

15 mittlere Tiefe der Querreihe (Soll 5,00 m): unser Code = 5.000000 | Bibliothek = 5.000000 -> ✅ gleich
15 mittlere Tiefe der Längsreihe (Soll 1,90 m): unser Code = 1.900000 | Bibliothek = 1.900000 -> ✅ gleich


**Ein Grenzfall ⚠️.** Steht ein Querfeld **allein** (2,5 × 5 m), ist seine lange Seite die Tiefe. Die Formel liefert dann 2,5 m und damit die Klasse `weit`. Der Schaden ist begrenzt: Die Fläche wird durch 10,66 m² statt durch 12,48 m² geteilt und zählt 1,17 statt 1,00 Platz-Äquivalente; gerundet bleibt es ein Platz.

In [20]:
einzelfeld = box(0, 0, 2.5, 5.0)                                       # ein einzelnes Querfeld
tiefe = mittlere_tiefe(einzelfeld)
klasse = "eng" if tiefe > 3.45 else "weit"                             # Grenze wie in Notebook 03
print(f"einzelnes Querfeld: mittlere Tiefe {tiefe:.2f} m → Klasse {klasse} | Platz-Äquivalente {einzelfeld.area / 10.66:.2f} statt {einzelfeld.area / 12.48:.2f}")
protokoll.append(dict(pruefung="15 Grenzfall: einzelnes Querfeld bekommt die Klasse weit", unser_code=tiefe, bibliothek=5.0, stimmt="⚠️"))

einzelnes Querfeld: mittlere Tiefe 2.50 m → Klasse weit | Platz-Äquivalente 1.17 statt 1.00


## 16. Platz-Äquivalente: summieren, dann einmal runden

**In Worten.** Jede gemeldete Fläche gibt Platz-Äquivalente: Fläche ÷ Platzfläche ihrer Klasse (12,48 m² für `eng`, 10,66 m² für `weit`; Notebook 03). Die Äquivalente eines Ausschnitts werden **summiert** und dann **einmal** gerundet.

**Prüfung.** Vier erfundene Flächen eines Ausschnitts, von Hand nachgerechnet: 24,9 ÷ 12,48 + 5,6 ÷ 12,48 + 5,6 ÷ 12,48 + 21,4 ÷ 10,66. Daneben steht, was herauskäme, wenn man jede Fläche einzeln rundete.

In [21]:
A_ENG, A_WEIT, GRENZE = 12.48, 10.66, 3.45                             # Platzflächen und Grenze der Tiefe (Notebook 03)
flaechen = pd.DataFrame({"flaeche_m2": [24.9, 5.6, 5.6, 21.4], "tiefe_m": [5.0, 4.6, 4.4, 1.9]})

# --- unser Code (wie in Notebook 09)
flaechen["klasse"] = np.where(flaechen.tiefe_m > GRENZE, "eng", "weit")
flaechen["aequivalente"] = flaechen.flaeche_m2 / flaechen.klasse.map({"eng": A_ENG, "weit": A_WEIT})
gezaehlt = int(np.floor(flaechen.aequivalente.sum() + 0.5))            # einmal je Ausschnitt runden

# --- von Hand
von_hand = 24.9 / 12.48 + 5.6 / 12.48 + 5.6 / 12.48 + 21.4 / 10.66     # 2,00 + 0,45 + 0,45 + 2,01 = 4,90
vergleiche("16 Platz-Äquivalente des Ausschnitts", flaechen.aequivalente.sum(), von_hand)
vergleiche("16 gezählt, einmal gerundet (Soll 5)", float(gezaehlt), 5.0)
einzeln = int(np.floor(flaechen.aequivalente + 0.5).sum())             # zum Vergleich: jede Fläche einzeln runden
print(f"einmal je Ausschnitt gerundet: {gezaehlt} Plätze | jede Fläche einzeln gerundet: {einzeln} Plätze (die zwei Bruchstücke gingen verloren)")
flaechen.round(2)

16 Platz-Äquivalente des Ausschnitts: unser Code = 4.900133 | Bibliothek = 4.900133 -> ✅ gleich
16 gezählt, einmal gerundet (Soll 5): unser Code = 5.000000 | Bibliothek = 5.000000 -> ✅ gleich
einmal je Ausschnitt gerundet: 5 Plätze | jede Fläche einzeln gerundet: 4 Plätze (die zwei Bruchstücke gingen verloren)


,flaeche_m2,tiefe_m,klasse,aequivalente
0,24.9,5.0,eng,2.00
1,5.6,4.6,eng,0.45
2,5.6,4.4,eng,0.45
3,21.4,1.9,weit,2.01


## 17. Bootstrap über ganze Blöcke

**In Worten.** Ausschnitte desselben Blocks zeigen dieselbe Strasse und sind nicht unabhängig. Notebook 10 zieht darum **ganze Blöcke** mit Zurücklegen und rechnet für jede Ziehung das Mittel über alle Ausschnitte der gezogenen Blöcke. Damit das schnell geht, rechnet es je Block vorab die Summe und die Anzahl: Mittel = Summe der gezogenen Summen ÷ Summe der gezogenen Anzahlen.

**Prüfung.** Dieselben 2'000 Ziehungen einmal mit dem schnellen Code und einmal Schritt für Schritt: die Ausschnitte der gezogenen Blöcke wirklich aneinanderhängen und mitteln.

In [22]:
werte = rng.normal(0.3, 1.0, 137)                                      # erfundener Wert je Ausschnitt (137 Ausschnitte)
block = rng.integers(0, 15, 137)                                       # jeder Ausschnitt gehört zu einem von 15 Blöcken

# --- unser Code (wie in Notebook 10, Funktion bootstrap_mittel)
je_block = pd.DataFrame({"wert": werte, "block": block}).groupby("block").wert.agg(["sum", "size"])
zufall = np.random.default_rng(42)
gezogen = zufall.integers(0, len(je_block), size=(2000, len(je_block)))        # Nummern der gezogenen Blöcke
schnell = je_block["sum"].values[gezogen].sum(axis=1) / je_block["size"].values[gezogen].sum(axis=1)

# --- zum Vergleich: dieselben Ziehungen Schritt für Schritt
langsam = []
for ziehung in gezogen:
    alle = np.concatenate([werte[block == je_block.index[nummer]] for nummer in ziehung])   # alle Ausschnitte der gezogenen Blöcke
    langsam.append(alle.mean())
vergleiche("17 grösster Unterschied zwischen schnellem und schrittweisem Bootstrap", float(np.abs(schnell - np.array(langsam)).max()), 0.0)
print("95-%-Intervall aus diesen Ziehungen:", np.percentile(schnell, [2.5, 97.5]).round(3))

17 grösster Unterschied zwischen schnellem und schrittweisem Bootstrap: unser Code = 0.000000 | Bibliothek = 0.000000 -> ✅ gleich
95-%-Intervall aus diesen Ziehungen: [0.156 0.436]


## 18. Zerlegung des Deltas in drei Teile

**In Worten.** Delta = Methode − amtlich. Notebook 10 zerlegt es mit der Ground Truth in drei Teile (`01b`, 12.9). Mit $M$ = gezählte Plätze der Methode, $S$ = sichtbare Felder, $V$ = verdeckt, belegt, $Z$ = einzeln stehend, $A$ = amtliche Zahl:

$$M - A = (M - S) \;-\; (V + Z) \;+\; (S + V + Z - A)$$

**Prüfung.** Ein Beispiel von Hand und 1'000 zufällige Fälle: Die drei Teile müssen zusammen immer genau das Delta ergeben.

In [23]:
M, S, V, Z, A = 90, 100, 12, 3, 95                                     # erfundene Zahlen
teil_1, teil_2, teil_3 = M - S, -(V + Z), S + V + Z - A                # −10, −15, +20
vergleiche("18 Summe der drei Teile (Soll: Delta = −5)", float(teil_1 + teil_2 + teil_3), float(M - A))

zahlen = rng.integers(0, 500, size=(1000, 5))                          # 1'000 zufällige Fälle: M, S, V, Z, A
m, s, v, z, a = zahlen.T
abweichend = int(((m - s) - (v + z) + (s + v + z - a) != m - a).sum())
vergleiche("18 zufällige Fälle, in denen die Teile nicht das Delta ergeben", float(abweichend), 0.0)

18 Summe der drei Teile (Soll: Delta = −5): unser Code = -5.000000 | Bibliothek = -5.000000 -> ✅ gleich
18 zufällige Fälle, in denen die Teile nicht das Delta ergeben: unser Code = 0.000000 | Bibliothek = 0.000000 -> ✅ gleich


## Ergebnis

Eine Zeile je Prüfung. ✅ = unsere Umsetzung stimmt mit der Bibliothek bzw. der Definition überein. ⚠️ = Abweichung gefunden, im Text erklärt.

(Bei Prüfung 8 steht in den Spalten die gemessene Abweichung und die erlaubte Toleranz, bei Prüfung 11 der kleinste und grösste Wert.)

In [24]:
ergebnis = pd.DataFrame(protokoll)
print("Prüfungen:", len(ergebnis), "| ✅", (ergebnis.stimmt == "✅").sum(), "| ⚠️", (ergebnis.stimmt == "⚠️").sum(), "| ❌", (ergebnis.stimmt == "❌").sum())
ergebnis.round(6)

Prüfungen: 38 | ✅ 37 | ⚠️ 1 | ❌ 0


,pruefung,unser_code,bibliothek,stimmt
0,1 IoU,0.356463,0.356463,✅
1,1 Genauigkeit,0.379161,0.379161,✅
2,1 Trefferquote,0.856209,0.856209,✅
3,2 MAE,1.593750,1.593750,✅
4,2 Bias in %,-20.053476,-20.053476,✅
5,3 Bootstrap untere Grenze,0.444359,0.447419,✅
6,3 Bootstrap obere Grenze,1.061571,1.071662,✅
7,4 Kreuzentropie,1.176514,1.176514,✅
8,"5 untere Grenze, 29 Lücken (Ziel 0.9)",0.901855,0.901855,✅
9,"5 untere Grenze, 59 Lücken (Ziel 0.95)",0.950492,0.950492,✅


**Was daraus folgt**
- Die Formeln dieses Durchgangs (`01b`, Abschnitt 12) sind so umgesetzt, wie sie dort stehen.
- **Platzfläche (Prüfung 12):** Mit der Normbreite zählt eine dicht stehende Reihe zu wenig; mit dem Abstand aus der Ground Truth geht die Rechnung auf. Das stützt den Entscheid in 01b, Abschnitt 1.
- **Zwei Rundungen (Prüfung 13):** Runden ist im Mittel neutral, Abrunden zählt einen halben Platz zu wenig – wie in 01b begründet.
- **Rundung (Prüfung 7):** Die Notebooks 05 und 09 runden mit `np.floor(x + 0.5)`; das ist kaufmännisch.
- **Grenzfall der Klasse (Prüfung 15):** Ein einzeln gemeldetes Querfeld bekommt die Klasse `weit`. Der Fehler ist begrenzt (1,17 statt 1,00 Platz-Äquivalente) und steht als Grenze in `01b`, 12.8.
- **Die Prüfungen 3, 5, 6, 8 und 11** gehören zum früheren Plan oder zum Konzept des Lückenschlusses. In diesem Durchgang wird damit nicht gerechnet.
- Dieses Notebook wird erweitert, sobald neue Formeln dazukommen.